# Análise exploratória da carteira de seguros

Este notebook responde a perguntas de negócio usando o data mart (`data/warehouse.db`) gerado pelo pipeline.
Cada seção parte de uma pergunta, consulta os dados com **SQL**, e termina com uma conclusão orientada à ação.

> Pré-requisito: `python -m insurance_analytics.pipeline`

In [1]:
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd
from scipy import stats

from insurance_analytics.charts import SERIES, _style
from insurance_analytics.config import PROJECT_ROOT

_style()
pd.set_option("display.float_format", "{:,.4f}".format)
con = sqlite3.connect(PROJECT_ROOT / "data" / "warehouse.db")
q = lambda sql, **p: pd.read_sql_query(sql, con, params=p)

## 1. Visão geral do data mart
Quantas linhas tem cada tabela do star schema?

In [2]:
tables = ["dim_date", "dim_customer", "dim_product", "dim_channel",
          "fact_policy", "fact_policy_month", "fact_claim", "fact_funnel_event"]
pd.DataFrame({"tabela": tables, "linhas": [q(f"SELECT COUNT(*) n FROM {t}")["n"][0] for t in tables]})

,tabela,linhas
0,dim_date,2922
1,dim_customer,20000
2,dim_product,6
3,dim_channel,3
4,fact_policy,28718
5,fact_policy_month,356167
6,fact_claim,4292
7,fact_funnel_event,308018


## 2. A sinistralidade do Auto é igual em todas as idades?
O Auto tem sinistralidade alta no agregado. Vale entender se o problema é concentrado em algum perfil.

In [3]:
auto = q('''
WITH premio AS (
    SELECT c.age_band, SUM(pm.earned_premium) AS premio
    FROM fact_policy_month pm JOIN dim_customer c ON c.customer_key = pm.customer_key
    WHERE pm.product_key = 3 AND pm.month_key >= 20250101
    GROUP BY c.age_band),
sin AS (
    SELECT c.age_band, SUM(CASE WHEN claim_status = 'aprovado' THEN paid_amount
                                WHEN claim_status = 'em_analise' THEN claim_amount ELSE 0 END) AS sinistro,
           COUNT(*) AS avisos
    FROM fact_claim fc JOIN dim_customer c ON c.customer_key = fc.customer_key
    WHERE fc.product_key = 3 AND fc.claim_date_key >= 20250101
    GROUP BY c.age_band)
SELECT p.age_band AS faixa_etaria, ROUND(p.premio, 2) AS premio_ganho, s.avisos,
       ROUND(s.sinistro / p.premio, 4) AS sinistralidade
FROM premio p LEFT JOIN sin s USING (age_band)
WHERE p.age_band <> 'ND'
ORDER BY p.age_band''')
auto

,faixa_etaria,premio_ganho,avisos,sinistralidade
0,18-24,"871,046.8700",35,0.7910
1,25-34,"1,473,763.1000",93,0.9769
2,35-44,"1,607,019.5000",58,0.4102
3,45-54,"1,012,647.1800",44,0.4608
4,55+,"480,746.7100",28,1.0981


In [4]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(auto["faixa_etaria"], auto["sinistralidade"], color=SERIES[0], width=0.6)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
ax.grid(axis="x", visible=False)
ax.set_title("Auto — sinistralidade por faixa etária (2025)");

**Leitura:** com poucos avisos por faixa, a variação entre faixas é em boa parte ruído — antes de mudar a precificação
por idade, seria necessário acumular mais exposição ou usar intervalos de confiança. Isso é um bom exemplo de por que olhar
o **volume de avisos** junto com a taxa.

## 3. Clientes que pagam com Pix cancelam mais? A diferença é estatisticamente significativa?
Hipótese de negócio: sem débito automático, o cliente precisa agir todo mês para pagar — e o atrito vira cancelamento.
Teste qui-quadrado de independência entre meio de pagamento e cancelamento em 12 meses.

In [5]:
canc = q('''
SELECT payment_method AS meio_pagamento,
       SUM(status = 'cancelada' AND tenure_days < 365) AS cancelou,
       SUM(NOT (status = 'cancelada' AND tenure_days < 365)) AS manteve
FROM fact_policy
WHERE start_date_key <= 20241231
GROUP BY payment_method''').set_index("meio_pagamento")
canc["taxa"] = canc["cancelou"] / (canc["cancelou"] + canc["manteve"])
chi2, p_value, dof, _ = stats.chi2_contingency(canc[["cancelou", "manteve"]])
print(f"qui² = {chi2:.1f} | gl = {dof} | p-valor = {p_value:.2e}")
canc

qui² = 113.7 | gl = 2 | p-valor = 1.99e-25


,cancelou,manteve,taxa
meio_pagamento,,,
cartao_credito,1696,5982,0.2209
debito_conta,1044,4327,0.1944
pix,698,1591,0.3049


**Leitura:** o p-valor é muito menor que 0,05, então a diferença não é acaso. Ação sugerida: oferecer benefício
(ex.: desconto na primeira parcela) para quem escolhe débito em conta na contratação, e medir o efeito com um teste A/B.

## 4. O ticket médio muda entre regiões?
Comparação do prêmio mensal das apólices ativas por região — teste de Kruskal-Wallis (não assume normalidade,
adequado para valores monetários assimétricos).

In [6]:
tk = q('''
SELECT c.region AS regiao, f.monthly_premium
FROM fact_policy f JOIN dim_customer c ON c.customer_key = f.customer_key
WHERE f.status = 'ativa' AND c.region <> 'ND' ''')
resumo = tk.groupby("regiao")["monthly_premium"].agg(apolices="size", media="mean", mediana="median")
h, p_value = stats.kruskal(*[g["monthly_premium"] for _, g in tk.groupby("regiao")])
print(f"Kruskal-Wallis H = {h:.2f} | p-valor = {p_value:.3f}")
resumo

Kruskal-Wallis H = 3.60 | p-valor = 0.463


,apolices,media,mediana
regiao,,,
Centro-Oeste,1659,62.0706,37.6900
Nordeste,3239,61.4041,38.2600
Norte,652,58.2997,38.2200
Sudeste,12677,60.5897,37.2100
Sul,3268,60.6746,37.0000


**Leitura:** p-valor bem acima de 0,05: não há evidência de diferença de ticket entre regiões — o mix de
produtos é parecido em todo o país, e segmentar campanhas por região não deve ser prioridade.

## 5. Quais fatores mais aumentam o risco de cancelamento?
Odds ratios da regressão logística (estimados no pipeline com `statsmodels`). Valor > 1 aumenta a chance de cancelar;
a referência de cada variável categórica é Vida / app / débito em conta / renda acima de 10 mil.

In [7]:
odds = pd.read_csv(PROJECT_ROOT / "reports" / "modelo_odds_ratios.csv")
sig = odds[odds["p_valor"] < 0.05].sort_values("odds_ratio")
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.errorbar(sig["odds_ratio"], sig["variavel"], xerr=[sig["odds_ratio"] - sig["ic95_inf"], sig["ic95_sup"] - sig["odds_ratio"]],
            fmt="o", color=SERIES[0], ecolor="#9a9a96", capsize=3)
ax.axvline(1, color="#52514e", lw=1, ls="--")
ax.set_xscale("log")
ax.set_title("Odds ratio de cancelamento (IC 95%, p < 0,05)");

**Leitura:** produto e meio de pagamento são os maiores drivers. Ter mais produtos no momento da contratação
**reduz** o risco — o cross-sell também é uma alavanca de retenção.

## Conclusões
1. **Retenção:** priorizar migração de Pix para débito automático — efeito grande e estatisticamente significativo.
2. **Cross-sell:** a maioria dos clientes tem um único produto, e cada produto adicional reduz a chance de cancelamento.
3. **Sinistralidade:** Pet e Auto estão acima de 60%; no Auto, as faixas 25-34 e 55+ puxam o indicador, mas confirmar o efeito exige mais volume de avisos.